# T35 - 不同曲线投资顺序分析
## 第3章：数据获取

### 概述
本章从数据库获取债券收益率曲线数据，包括：
1. 数据库连接与测试
2. 构建SQL查询语句
3. 获取利率债收益率数据
4. 获取信用债收益率数据
5. 数据清洗与预处理
6. 保存收益率数据

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import os
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("库导入成功！")

In [ ]:
# 2. 加载配置
from config import (
    get_database_url,
    START_DATE, END_DATE,
    BOND_TYPES, RATE_BONDS, FINANCE_BONDS, CREDIT_BONDS,
    RATE_BOND_TERMS, CREDIT_BOND_TERMS,
    RATINGS,
    CLASSIFY2_MAPPING,
    YIELD_DATA_FILE,
    CYCLE_LABEL_FILE,
    OUTPUT_DIR
)

print(f"日期范围: {START_DATE} ~ {END_DATE}")
print(f"输出文件: {YIELD_DATA_FILE}")

In [ ]:
# 3. 建立数据库连接
def create_db_connection():
    """创建数据库连接"""
    try:
        db_url = get_database_url()
        engine = create_engine(
            db_url,
            poolclass=sqlalchemy.pool.NullPool,
            pool_recycle=3600
        )
        
        # 测试连接
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        
        print("数据库连接成功！")
        return engine
    except Exception as e:
        print(f"数据库连接失败: {e}")
        print("\n提示: 如果是密码问题，请设置环境变量 DB_PASSWORD")
        print("\n将使用模拟数据进行演示...")
        return None

db_engine = create_db_connection()

In [ ]:
# 4. SQL查询构建函数

def build_rate_bond_query(bond_type, start_date, end_date, term_groups):
    """
    构建利率债收益率查询SQL
    
    Parameters:
    -----------
    bond_type : str
        债券类型（国债、国开、口行、农发、地方债）
    start_date : str
        开始日期
    end_date : str
        结束日期
    term_groups : list
        期限分组列表
    
    Returns:
    --------
    str
        SQL查询语句
    """
    classify2 = CLASSIFY2_MAPPING.get(bond_type, '')
    
    # 构建期限分组CASE语句
    term_case = """
    CASE 
        WHEN sq.曲线期限 < 1 THEN '0-1'
        WHEN sq.曲线期限 < 3 THEN '1-3'
        WHEN sq.曲线期限 < 5 THEN '3-5'
        WHEN sq.曲线期限 < 7 THEN '5-7'
        WHEN sq.曲线期限 < 10 THEN '7-10'
        WHEN sq.曲线期限 < 13 THEN '10-13'
        WHEN sq.曲线期限 < 17 THEN '13-17'
        WHEN sq.曲线期限 < 23 THEN '17-23'
        WHEN sq.曲线期限 < 27 THEN '23-27'
        WHEN sq.曲线期限 < 33 THEN '27-33'
        ELSE '33-50'
    END as 久期
    """
    
    query = f"""
    SELECT 
        dt,
        '{bond_type}' AS 曲线名称,
        avg(收益率) as 收益率,
        {term_case}
    FROM (
        SELECT 
            A.dt,
            LEFT(SUBSTRING_INDEX(B.SEC_NAME, ':', -1), 
                CHAR_LENGTH(SUBSTRING_INDEX(B.SEC_NAME, ':', -1)) - 1) * (
                (RIGHT(B.SEC_NAME, 1) = '天') / 365 + 
                (RIGHT(B.SEC_NAME, 1) = '月') / 12 + 
                (RIGHT(B.SEC_NAME, 1) = '年')
            ) AS 曲线期限,
            A.CLOSE AS 收益率
        FROM bond.marketinfo_curve A
        INNER JOIN bond.basicinfo_curve B ON A.trade_code = B.TRADE_CODE
        WHERE A.dt <= '{end_date}' AND A.dt >= '{start_date}'
        AND B.classify2 = '{classify2}'
    ) sq
    WHERE sq.曲线期限 > 0
    GROUP BY dt, 曲线名称, 久期
    """
    
    return query


def build_credit_bond_query(bond_type, start_date, end_date, ratings):
    """
    构建信用债收益率查询SQL
    
    Parameters:
    -----------
    bond_type : str
        债券类型（城投、产业、普通金融债、二永、存单）
    start_date : str
        开始日期
    end_date : str
        结束日期
    ratings : list
        评级列表
    
    Returns:
    --------
    str
        SQL查询语句
    """
    classify2 = CLASSIFY2_MAPPING.get(bond_type, '')
    ratings_str = "','".join(ratings)
    
    # 构建期限分组CASE语句（信用债）
    term_case = """
    CASE 
        WHEN sq.曲线期限 < 1.5 THEN '0-1'
        WHEN sq.曲线期限 < 3.5 THEN '2-3'
        WHEN sq.曲线期限 < 5.5 THEN '4-5'
        WHEN sq.曲线期限 < 7.5 THEN '6-7'
        WHEN sq.曲线期限 < 12.5 THEN '8-12'
        WHEN sq.曲线期限 < 17.5 THEN '13-17'
        WHEN sq.曲线期限 < 22.5 THEN '18-22'
        WHEN sq.曲线期限 < 27.5 THEN '23-27'
        ELSE '28-50'
    END as 久期
    """
    
    query = f"""
    SELECT 
        dt,
        '{bond_type}' AS 曲线名称,
        avg(收益率) as 收益率,
        {term_case}
    FROM (
        SELECT 
            A.dt,
            LEFT(SUBSTRING_INDEX(B.SEC_NAME, ':', -1), 
                CHAR_LENGTH(SUBSTRING_INDEX(B.SEC_NAME, ':', -1)) - 1) * (
                (RIGHT(B.SEC_NAME, 1) = '天') / 365 + 
                (RIGHT(B.SEC_NAME, 1) = '月') / 12 + 
                (RIGHT(B.SEC_NAME, 1) = '年')
            ) AS 曲线期限,
            SUBSTRING(REPLACE(B.classify2, '＋', '+'), 
                LOCATE('(', REPLACE(B.classify2, '＋', '+')) + 1,
                CHAR_LENGTH(B.classify2) - LOCATE('(', REPLACE(B.classify2, '＋', '+')) - 1
            ) AS 隐含评级,
            A.CLOSE AS 收益率
        FROM bond.marketinfo_curve A
        INNER JOIN bond.basicinfo_curve B ON A.trade_code = B.TRADE_CODE
        WHERE A.dt <= '{end_date}' AND A.dt >= '{start_date}'
        AND B.classify2 LIKE '{classify2}%%'
    ) sq
    WHERE sq.隐含评级 IN ('{ratings_str}')
    AND sq.曲线期限 > 0
    GROUP BY dt, 曲线名称, 久期
    """
    
    return query

print("SQL查询构建函数已定义")

In [ ]:
# 5. 数据获取函数

def fetch_yield_data_from_db(engine):
    """从数据库获取所有债券收益率数据"""
    if engine is None:
        return None
    
    all_data = []
    
    print("开始获取债券收益率数据...")
    print("=" * 60)
    
    # 1. 获取利率债数据
    print("\n1. 获取利率债数据...")
    for bond_type in RATE_BONDS:
        print(f"   - {bond_type}...", end=" ")
        try:
            query = build_rate_bond_query(bond_type, START_DATE, END_DATE, RATE_BOND_TERMS)
            df = pd.read_sql(query, engine)
            if not df.empty:
                all_data.append(df)
                print(f"获取 {len(df)} 条记录")
            else:
                print("无数据")
        except Exception as e:
            print(f"失败: {e}")
    
    # 2. 获取信用债数据
    print("\n2. 获取信用债数据...")
    for bond_type in CREDIT_BONDS + FINANCE_BONDS:
        print(f"   - {bond_type}...", end=" ")
        try:
            query = build_credit_bond_query(bond_type, START_DATE, END_DATE, RATINGS)
            df = pd.read_sql(query, engine)
            if not df.empty:
                all_data.append(df)
                print(f"获取 {len(df)} 条记录")
            else:
                print("无数据")
        except Exception as e:
            print(f"失败: {e}")
    
    # 合并所有数据
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n总计获取 {len(combined_df)} 条记录")
        return combined_df
    else:
        print("\n未获取到任何数据")
        return None

# 尝试从数据库获取数据
if db_engine is not None:
    raw_data = fetch_yield_data_from_db(db_engine)
else:
    raw_data = None

In [ ]:
# 6. 生成模拟数据（当数据库不可用时）

def generate_mock_yield_data():
    """生成模拟的债券收益率数据（用于演示）"""
    print("生成模拟债券收益率数据...")
    print("=" * 60)
    
    # 读取周期标注
    if os.path.exists(CYCLE_LABEL_FILE):
        cycles_df = pd.read_csv(CYCLE_LABEL_FILE, encoding='utf-8-sig')
    else:
        # 使用默认周期
        cycles_df = pd.DataFrame([
            {'cycle': 1, 'start_date': '2014-01-02', 'end_date': '2015-02-13', 
             'start_yield': 4.6018, 'end_yield': 3.3552, 'yield_change': -1.2466},
            {'cycle': 2, 'start_date': '2015-06-11', 'end_date': '2016-01-11',
             'start_yield': 3.6197, 'end_yield': 2.8187, 'yield_change': -0.8010},
            {'cycle': 3, 'start_date': '2018-01-18', 'end_date': '2019-02-02',
             'start_yield': 3.9797, 'end_yield': 3.0978, 'yield_change': -0.8819},
            {'cycle': 4, 'start_date': '2019-11-04', 'end_date': '2020-04-21',
             'start_yield': 3.2954, 'end_yield': 2.5791, 'yield_change': -0.7163},
            {'cycle': 5, 'start_date': '2021-03-10', 'end_date': '2022-08-29',
             'start_yield': 3.2353, 'end_yield': 2.6225, 'yield_change': -0.6128},
            {'cycle': 6, 'start_date': '2023-02-22', 'end_date': '2024-08-20',
             'start_yield': 2.9175, 'end_yield': 2.1710, 'yield_change': -0.7465}
        ])
    
    # 基础利差设置
    term_spreads = {
        '短期': -1.0,
        '中长期': -0.3,
        '长期': 0.0,
        '超长期': 0.3
    }
    
    credit_spreads = {
        'AAA': 0.5,
        'AA+': 0.8,
        'AA': 1.2,
        'AA(2)': 1.8,
        'AA-': 2.2
    }
    
    data = []
    
    for _, cycle in cycles_df.iterrows():
        start_date = pd.Timestamp(cycle['start_date'])
        end_date = pd.Timestamp(cycle['end_date'])
        dates = pd.date_range(start_date, end_date, freq='B')
        total_days = (end_date - start_date).days
        
        for date in dates:
            progress = (date - start_date).days / total_days
            current_yield = cycle['start_yield'] + progress * cycle['yield_change']
            
            # 生成国债收益率
            for term, spread in term_spreads.items():
                base_yield = current_yield + spread + np.random.normal(0, 0.02)
                data.append({
                    'dt': date,
                    'bond_type': '国债',
                    'term': term,
                    'rating': None,
                    'yield_rate': max(base_yield, 0.5),
                    'cycle_id': cycle['cycle']
                })
                
                # 生成金融债
                if term != '超长期':
                    for bond_type in ['二永', '存单']:
                        for rating, credit_spread in credit_spreads.items():
                            yield_rate = base_yield + credit_spread + np.random.normal(0, 0.03)
                            data.append({
                                'dt': date,
                                'bond_type': bond_type,
                                'term': term,
                                'rating': rating,
                                'yield_rate': max(yield_rate, 0.8),
                                'cycle_id': cycle['cycle']
                            })
                
                # 生成城投债
                if term != '超长期':
                    for rating, credit_spread in credit_spreads.items():
                        yield_rate = base_yield + credit_spread * 1.3 + np.random.normal(0, 0.04)
                        data.append({
                            'dt': date,
                            'bond_type': '城投',
                            'term': term,
                            'rating': rating,
                            'yield_rate': max(yield_rate, 1.0),
                            'cycle_id': cycle['cycle']
                        })
    
    df = pd.DataFrame(data)
    df = df.groupby(['dt', 'bond_type', 'term', 'rating', 'cycle_id'])['yield_rate'].mean().reset_index()
    df = df.sort_values('dt')
    
    print(f"生成 {len(df)} 条模拟数据记录")
    return df

# 如果数据库不可用，使用模拟数据
if raw_data is None:
    print("\n数据库不可用，使用模拟数据进行演示...")
    yield_data = generate_mock_yield_data()
else:
    # 转换数据格式
    yield_data = raw_data.copy()
    yield_data = yield_data.rename(columns={'曲线名称': 'bond_type', '久期': 'term', '收益率': 'yield_rate'})
    yield_data['rating'] = 'NA'  # 利率债没有评级
    
    # 添加周期ID
    yield_data['cycle_id'] = 0
    for _, cycle in pd.read_csv(CYCLE_LABEL_FILE).iterrows():
        mask = (yield_data['dt'] >= cycle['start_date']) & (yield_data['dt'] <= cycle['end_date'])
        yield_data.loc[mask, 'cycle_id'] = cycle['cycle']

print(f"\n数据准备完成，共 {len(yield_data)} 条记录")

In [ ]:
# 7. 数据清洗与预处理

def clean_yield_data(df):
    """清洗收益率数据"""
    print("开始数据清洗...")
    print("=" * 60)
    
    print(f"\n原始数据: {len(df)} 条记录")
    
    # 1. 转换日期格式
    df['dt'] = pd.to_datetime(df['dt'])
    
    # 2. 检查缺失值
    print("\n缺失值统计:")
    for col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            print(f"   {col}: {missing} ({missing/len(df)*100:.2f}%)")
    
    # 3. 筛选指定期限
    valid_terms = ['0-1', '1-3', '2-3', '4-5', '6-7', '8-12', '13-17', '18-22', '23-27', '28-50',
                   '短期', '中短期', '中长期', '长期', '超长期']
    df = df[df['term'].isin(valid_terms)]
    print(f"\n期限筛选后: {len(df)} 条记录")
    
    # 4. 处理异常值
    df = df[df['yield_rate'] > 0]  # 收益率必须为正
    df = df[df['yield_rate'] < 20]  # 收益率不超过20%
    print(f"异常值处理后: {len(df)} 条记录")
    
    # 5. 按日期排序
    df = df.sort_values(['dt', 'bond_type', 'term']).reset_index(drop=True)
    
    return df

# 清洗数据
yield_data = clean_yield_data(yield_data)

print("\n数据清洗完成！")

In [ ]:
# 8. 数据统计与预览

def display_data_statistics(df):
    """显示数据统计信息"""
    print("数据统计")
    print("=" * 60)
    
    # 1. 基本统计
    print("\n1. 基本统计:")
    print(f"   总记录数: {len(df):,}")
    print(f"   日期范围: {df['dt'].min().strftime('%Y-%m-%d')} ~ {df['dt'].max().strftime('%Y-%m-%d')}")
    print(f"   交易日数: {df['dt'].nunique()}")
    
    # 2. 按债券类型统计
    print("\n2. 按债券类型统计:")
    type_stats = df.groupby('bond_type').agg({
        'yield_rate': ['count', 'mean', 'std', 'min', 'max']
    }).round(4)
    type_stats.columns = ['记录数', '平均收益率', '标准差', '最小值', '最大值']
    print(type_stats.to_string())
    
    # 3. 按期限统计
    print("\n3. 按期限统计:")
    term_stats = df.groupby('term').agg({
        'yield_rate': ['count', 'mean', 'std']
    }).round(4)
    term_stats.columns = ['记录数', '平均收益率', '标准差']
    print(term_stats.to_string())
    
    # 4. 按周期统计
    print("\n4. 按周期统计:")
    cycle_stats = df.groupby('cycle_id').agg({
        'dt': ['min', 'max', 'nunique'],
        'yield_rate': ['count', 'mean']
    })
    cycle_stats.columns = ['起始日期', '结束日期', '交易日数', '记录数', '平均收益率']
    print(cycle_stats.to_string())
    
    # 5. 数据预览
    print("\n5. 数据预览（前10条）:")
    print(df.head(10).to_string(index=False))

display_data_statistics(yield_data)

In [ ]:
# 9. 保存数据

def save_yield_data(df):
    """保存收益率数据"""
    # 转换日期格式
    df_to_save = df.copy()
    df_to_save['dt'] = df_to_save['dt'].dt.strftime('%Y-%m-%d')
    
    # 保存CSV
    df_to_save.to_csv(YIELD_DATA_FILE, index=False, encoding='utf-8-sig', float_format='%.4f')
    
    print(f"数据已保存: {YIELD_DATA_FILE}")
    print(f"文件大小: {os.path.getsize(YIELD_DATA_FILE):,} bytes")

save_yield_data(yield_data)

In [ ]:
# 10. 验证保存的数据

def verify_saved_data():
    """验证保存的数据"""
    print("验证保存的数据")
    print("=" * 60)
    
    if os.path.exists(YIELD_DATA_FILE):
        df = pd.read_csv(YIELD_DATA_FILE, encoding='utf-8-sig')
        print(f"\n文件: {os.path.basename(YIELD_DATA_FILE)}")
        print(f"记录数: {len(df):,}")
        print(f"列名: {list(df.columns)}")
        print(f"\n前5条记录:")
        print(df.head().to_string(index=False))
        print(f"\n后5条记录:")
        print(df.tail().to_string(index=False))
    else:
        print(f"错误: 文件不存在 - {YIELD_DATA_FILE}")

verify_saved_data()

### 总结

本章完成了以下工作：
1. 建立了数据库连接（或生成模拟数据）
2. 构建了SQL查询语句获取债券收益率数据
3. 获取了利率债和信用债的收益率数据
4. 进行了数据清洗与预处理
5. 保存了收益率数据文件

**输出文件:**
- `债券收益率数据.csv`: 包含日期、债券类型、期限、评级、收益率、周期ID

**下一步**: 运行 `04-策略生成.ipynb` 生成投资策略组合。